<a href="https://colab.research.google.com/github/snehasajjan421-ops/Plant-Disease-Classification/blob/main/Plant__Disease_Image_Classification_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [4]:
# Update these paths to match your folder structure in Google Drive
train_dir = train_dir = '/content/PlantDiseaseProject/train'
test_dir = test_dir = '/content/PlantDiseaseProject/test'
print("Train path exists:", os.path.exists(train_dir))
print("Test path exists:", os.path.exists(test_dir))

Train path exists: True
Test path exists: True


In [5]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [6]:
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

print(f"Total training images: {len(train_dataset)}")
print(f"Total testing images: {len(test_dataset)}")
print(f"Classes found ({len(train_dataset.classes)}): {train_dataset.classes}")

Total training images: 4360
Total testing images: 1092
Classes found (4): ['Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy']


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("PyTorch DataLoaders are ready!")

PyTorch DataLoaders are ready!


In [8]:
from torchvision import models

# Load a pretrained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze early layers so their pre-learned weights don't change
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer to match your number of classes
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Move model to GPU (or CPU)
model = model.to(device)

print(f"Model configured for {num_classes} classes and moved to {device}.")

Model configured for 4 classes and moved to cpu.


In [31]:
import torch.optim as optim

# CrossEntropyLoss is standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Only train the weights of the final layer (model.fc)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Loss function and Optimizer are ready!")

Loss function and Optimizer are ready!


In [9]:
# Load the saved model weights from Google Drive
save_path = '/content/drive/MyDrive/plant_disease_model.pth'
model.load_state_dict(torch.load(save_path, map_location=device))
print("Trained model weights loaded successfully!")

Trained model weights loaded successfully!


In [33]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Calculate training statistics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"Epoch [{epoch + 1}/{num_epochs}] - Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

Epoch [1/5] - Loss: 0.3813 | Accuracy: 88.58%
Epoch [2/5] - Loss: 0.2857 | Accuracy: 90.92%
Epoch [3/5] - Loss: 0.2414 | Accuracy: 92.45%
Epoch [4/5] - Loss: 0.2199 | Accuracy: 92.61%
Epoch [5/5] - Loss: 0.2053 | Accuracy: 93.42%


In [35]:
model.eval()  # Put the model in evaluation mode
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():  # Turn off gradient tracking for faster inference
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

final_test_acc = (correct / total) * 100
final_test_loss = test_loss / total

print(f"Final Test Loss: {final_test_loss:.4f}")
print(f"Final Test Accuracy: {final_test_acc:.2f}%")

Final Test Loss: 0.1598
Final Test Accuracy: 95.70%


In [36]:
# Save model weights to Google Drive
save_path = '/content/drive/MyDrive/plant_disease_model.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved successfully to: {save_path}")

Model saved successfully to: /content/drive/MyDrive/plant_disease_model.pth


In [37]:
import json

class_names_path = '/content/drive/MyDrive/class_names.json'
with open(class_names_path, 'w') as f:
    json.dump(train_dataset.classes, f)

print(f"Class labels saved successfully to {class_names_path}!")

Class labels saved successfully to /content/drive/MyDrive/class_names.json!


In [38]:
!pip install gradio -q

In [40]:
!pip install --upgrade huggingface_hub gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.0 MB/s eta 0:00:00


In [10]:
import gradio as gr
import torch
from PIL import Image

model.eval()

def classify_leaf(input_image):
    if input_image is None:
        return "Please upload an image."

    image = Image.fromarray(input_image).convert('RGB')
    tensor_img = test_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor_img)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    confidences = {train_dataset.classes[i]: float(probabilities[i]) for i in range(len(train_dataset.classes))}
    return confidences

interface = gr.Interface(
    fn=classify_leaf,
    inputs=gr.Image(label="Upload Leaf Image"),
    outputs=gr.Label(num_top_classes=3, label="Predicted Disease"),
    title="🌱 Plant Disease Classification System",
    description="Upload an image of a plant leaf to identify the disease and confidence score."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46345f9aa336d2b320.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
import json

with open('/content/drive/MyDrive/class_names.json', 'w') as f:
    json.dump(train_dataset.classes, f)

print("Classes saved successfully!")

Classes saved successfully!


In [12]:
!pip install --upgrade gradio torchvision -q

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json
import gradio as gr
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Set Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Load saved classes
with open('/content/drive/MyDrive/class_names.json', 'r') as f:
    class_names = json.load(f)

# 4. Recreate model architecture and load saved weights
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(torch.load('/content/drive/MyDrive/plant_disease_model.pth', map_location=device))
model = model.to(device)
model.eval()

# 5. Define transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 6. Prediction function & Interface
def classify_leaf(input_image):
    if input_image is None:
        return "Upload an image."
    img = Image.fromarray(input_image).convert('RGB')
    tensor_img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(tensor_img)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

gr.Interface(
    fn=classify_leaf,
    inputs=gr.Image(label="Upload Leaf"),
    outputs=gr.Label(num_top_classes=3, label="Prediction"),
    title="🌱 Plant Disease Classification System"
).launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 779.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.